# Classifier Ablation Studies

In [1]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"

In [2]:
%load_ext autoreload
import sys, time, json, copy
import functools, warnings
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

In [3]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [4]:
%autoreload 2
from lib.graph_factory import GraphFactory, add_graph_plot
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed, save_to_pkl, load_from_pkl
from lib.classifier import ByzClassifier, load_run, save_run
from lib.system import SystemSimulator
from lib.metrics import MetricsCalculator
from lib.config import BASE_CONF, NUM_NODES
from lib.simulation import load_dataset
from lib.preprocessor import FEATURE_NAMES, HEAVY_FEATURES

**Experiment Configuration**

In [ ]:
RUN_DIR = os.path.join(Path().resolve(), "baseline")
IMAGES_DIR = os.path.join(RUN_DIR, "images")
config = copy.deepcopy(BASE_CONF)

# Globals
b = 5

# Target stable Distribution
# config['pi'] = (np.full(NUM_NODES, 1.0/NUM_NODES))
config['pi-dir-alpha'] = 10
pi = rng('dir','graphs').dirichlet(np.full(NUM_NODES,config['pi-dir-alpha']))
config['pi'] = (pi / pi.sum())

# Classifier Training 
config['train']['b'] = b
config['train']['fpr_mean'] = 0.2
config['train']['fpr_spread'] = 0.05
config['train']['gamma_C'] = np.clip((config['train']['fpr_spread']*np.linspace(-1,1,NUM_NODES)) + \
                                     config['train']['fpr_mean'], 0, 0.95)

# System Simulation 
config['sys']['b'] = b
config['sys']['fpr_spread'] = 0.003

# Communication Graph
config['graph_type'] = 'erdos-renyi'
config['graph_weights'] = 'MH_gen'
config['graph_args'] = {
    'ba_m': 5,
    'ws_k': 6,
    'ws_p': 0.1,
    'rand_reg_deg':b+1,
    'geom_radius':0.6,
    'er_p':min(1.0, 3.0 * np.log(NUM_NODES) / NUM_NODES),
}
config['graph_args']['MH_target_pi'] = (1 - config['train']['gamma_C']) * config['pi']

# Miscellaneous
config['train']['clf_model'] = 'xgb'
config['data_heterogeneity'] = 100
config['reg_param'] = 1

# Notebook Variables
ATKS = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
ALGS = ['RDSGD', 'ORACLE', 'IOS', 'SCC', 'TriMean', 'CooMed']
ALGS_TEST = ['RDSGD', 'ORACLE']
COLORS = dict(zip(ALGS, ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']))
LOCAL_SEED = 777
DEBUG = True

In [6]:
# Save experiment configuration
payload = copy.deepcopy(config)
del payload['train']['gamma_C']
del payload['pi']
del payload['graph_args']['MH_target_pi']
with open(os.path.join(RUN_DIR, f'config.json'), 'w') as f:
    def convert(o):
        if isinstance(o, np.generic):
            return o.item()
        raise TypeError(f'{type(o)} not serializable')
    json.dump(payload, f, indent=2, default=convert)

In [7]:
# Initialize Globals
gd = load_dataset()
gf = GraphFactory(config['train']['num_nodes'], b)

In [8]:
# Estimate Projection Constant
pce = ProjConstEstimator(config, gd, gf)
pce.configure(config, seed('proj-const-estimator'))
proj_const = pce.estimate()

### Sensitivity Analysis

In [ ]:
sep, imp = {}, {}
for atk in ATKS:
    clf = ByzClassifier(config, gd, gf)
    clf.init_preproc()
    clf.init_simulation(config, proj_const, LOCAL_SEED)
    Xtr, ytr, gtr = clf.simulate(atk, LOCAL_SEED, config['train']['beta_C'], config['train']['gamma_C'])

    # (a) univariate: 2*AUC-1 in [-1,1], sign = direction of the shift
    sep[atk] = {FEATURE_NAMES[f]: 2 * roc_auc_score(ytr, Xtr[:, f]) - 1
                for f in range(Xtr.shape[1])}

    # (b) multivariate, on a held-out simulation
    Xtr_s = clf.feat_pre_proc.fit_transform(Xtr)
    est, _ = clf.fit(in_data=dict(X_train=Xtr_s, y_train=ytr, groups_train=gtr))

    clf.init_simulation(config, proj_const, VAL_SEED)
    Xva, yva, _ = clf.simulate(atk, VAL_SEED,
                               config['train']['beta_C'], config['train']['gamma_C'])
    r = permutation_importance(est, clf.feat_pre_proc.transform(Xva), yva,
                               scoring='average_precision', n_repeats=10,
                               random_state=seed('perm', atk))
    imp[atk] = dict(zip(clf.out_feat_names, r.importances_mean))   # output order — correct here

sep = pd.DataFrame(sep).T.reindex(columns=FEATURE_NAMES)
imp = pd.DataFrame(imp).T.reindex(columns=FEATURE_NAMES)

In [ ]:
display(sep)

In [ ]:
display(imp)

### Classifier Model Selection

### Leave-One-Out Analysis

### Train-Val Matrix